# Minimum Variance Portfolio Optimization — Version 2
## Rolling Backtest and Robust Portfolio Framework

**Author:** Thomas Christian Matenco  
**Course:** Algorithmic Trading, IE University MBDS  
**Version:** 2.0 — Rolling Backtest Extension

---

This notebook extends the single-period Version 1 analysis into a full rolling-window backtest framework.

**What Version 2 adds over Version 1:**
- Rolling-window backtesting across multiple out-of-sample years (2019–2023)
- Four portfolio construction strategies compared side by side
- Rebalancing logic (annual, quarterly, monthly)
- Transaction cost modelling
- Risk contribution analysis
- Extended performance metrics (Sortino, Calmar, VaR, CVaR, Hit Rate)
- Full visualisation suite

**Assets:** ACWI (global equities) | AGG (US aggregate bonds) | GLD (gold) | BSV (short-term bonds)  
**Benchmark:** Equal-weight Permanent Portfolio (25% each)  
**Training window:** 4 years, rolling  
**Test periods:** 2019, 2020, 2021, 2022, 2023  
**Risk-free rate:** 0% (consistent with Version 1)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
import warnings

from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_columns", 20)

TRADING_DAYS = 252
RISK_FREE    = 0.00
TICKERS      = ["ACWI", "AGG", "GLD", "BSV"]
MAX_WEIGHT   = 0.40
TRAIN_YEARS  = 4
TEST_YEARS   = [2019, 2020, 2021, 2022, 2023]
TC_BPS       = 5   # transaction cost in basis points per unit of turnover

COLORS = {
    "min_variance":  "#111111",
    "risk_parity":   "#555555",
    "max_sharpe":    "#999999",
    "equal_weight":  "#CCCCCC",
    "benchmark":     "#000000",
}
LABELS = {
    "min_variance": "Min Variance (40% cap)",
    "risk_parity":  "Risk Parity",
    "max_sharpe":   "Max Sharpe",
    "equal_weight": "Equal Weight (Benchmark)",
}

## 2. Data Download

We extend the data window back to 2015 to support rolling backtests starting in 2019 (requiring 4 years of training data).

**Training cutoff convention:** each test year uses data up to December 28 of the prior year, consistent with the Version 1 submission date (2022-12-29 effective date).

In [ ]:
prices = yf.download(
    TICKERS,
    start="2015-01-01",
    end="2024-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices.reindex(columns=TICKERS).dropna()

assert list(prices.columns) == TICKERS, "Column order mismatch"

print(f"Downloaded: {prices.shape[0]} trading days")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nFirst rows:")
display(prices.head())

In [ ]:
returns = prices.pct_change().dropna()

# Verify no look-ahead: each test year's training ends on Dec 28 of prior year
for yr in TEST_YEARS:
    train_end = f"{yr-1}-12-28"
    test_start = f"{yr}-01-01"
    n_train = len(returns.loc[f"{yr-TRAIN_YEARS}-01-01":train_end])
    n_test  = len(returns.loc[test_start:f"{yr}-12-31"])
    print(f"  {yr}  |  train obs: {n_train:4d}  |  test obs: {n_test:3d}")

print(f"\nTotal returns shape: {returns.shape}")

In [ ]:
# Normalized price performance (full period)
norm = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(13, 5))
line_styles = {"ACWI": "-", "AGG": "--", "GLD": "-.", "BSV": ":"}
for t in TICKERS:
    ax.plot(norm.index, norm[t], label=t, linewidth=1.8, linestyle=line_styles[t], color=COLORS.get(t, "#333333"))

# Mark each test-year start
for yr in TEST_YEARS:
    ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="grey", linewidth=0.7, linestyle="--", alpha=0.5)

ax.axvline(pd.Timestamp("2019-01-01"), color="black", linewidth=1.0, linestyle="--", label="First test period (2019)")
ax.set_title("Normalized Price Performance (Base = 100, Jan 2015)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Index Value")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Core Functions

All optimization and analysis functions are defined here once and reused throughout the notebook.

In [ ]:
# ── Covariance estimation ─────────────────────────────────────────

def ledoit_wolf_cov(returns_df):
    """Estimate annualised Ledoit-Wolf shrinkage covariance matrix."""
    lw = LedoitWolf().fit(returns_df.values)
    cov_ann = lw.covariance_ * TRADING_DAYS
    return cov_ann, lw.shrinkage_


# ── Portfolio optimizers ──────────────────────────────────────────

def minimize_variance(cov_matrix, max_weight=MAX_WEIGHT):
    """Long-only minimum variance portfolio with per-asset weight cap."""
    n = cov_matrix.shape[0]
    result = minimize(
        lambda w: np.sqrt(w.T @ cov_matrix @ w),
        x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, max_weight)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success:
        raise ValueError(f"MinVar failed: {result.message}")
    return result.x


def maximize_sharpe(exp_returns_ann, cov_matrix, rf=RISK_FREE):
    """Long-only maximum Sharpe portfolio."""
    n = len(exp_returns_ann)
    def neg_sharpe(w):
        ret = w @ exp_returns_ann
        vol = np.sqrt(w.T @ cov_matrix @ w)
        return -(ret - rf) / vol if vol > 1e-10 else 1e10
    result = minimize(
        neg_sharpe, x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success:
        raise ValueError(f"MaxSharpe failed: {result.message}")
    return result.x


def risk_contributions(weights, cov_matrix):
    """Percentage contribution of each asset to total portfolio risk."""
    port_vol = np.sqrt(weights.T @ cov_matrix @ weights)
    if port_vol < 1e-10:
        return np.zeros_like(weights)
    marginal = cov_matrix @ weights / port_vol
    rc = weights * marginal
    return rc / rc.sum()


def risk_parity_objective(weights, cov_matrix):
    """Sum of squared deviations from equal risk contribution."""
    rc = risk_contributions(weights, cov_matrix)
    target = np.repeat(1 / len(weights), len(weights))
    return np.sum((rc - target) ** 2)


def optimize_risk_parity(cov_matrix, max_weight=1.0):
    """Long-only risk parity (equal risk contribution) portfolio."""
    n = cov_matrix.shape[0]
    result = minimize(
        lambda w: risk_parity_objective(w, cov_matrix),
        x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, max_weight)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success:
        raise ValueError(f"RiskParity failed: {result.message}")
    return result.x


# ── Performance metrics ───────────────────────────────────────────

def performance_metrics(daily_returns, rf=RISK_FREE, label=None):
    """Full performance metric suite for a daily return series."""
    ann_ret = daily_returns.mean() * TRADING_DAYS
    ann_vol = daily_returns.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - rf) / ann_vol if ann_vol > 0 else np.nan

    downside = daily_returns[daily_returns < 0]
    dv = downside.std() * np.sqrt(TRADING_DAYS) if len(downside) > 1 else np.nan
    sortino = (ann_ret - rf) / dv if dv and dv > 0 else np.nan

    cum = (1 + daily_returns).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar = ann_ret / abs(mdd) if mdd != 0 else np.nan

    var95  = daily_returns.quantile(0.05)
    cvar95 = daily_returns[daily_returns <= var95].mean()

    return {
        "Ann. Return":    ann_ret,
        "Ann. Volatility":ann_vol,
        "Sharpe":         sharpe,
        "Sortino":        sortino,
        "Calmar":         calmar,
        "Max Drawdown":   mdd,
        "Cum. Return":    cum.iloc[-1] - 1,
        "VaR 95%":        var95,
        "CVaR 95%":       cvar95,
        "Hit Rate":       (daily_returns > 0).mean(),
    }


# ── Turnover and transaction costs ───────────────────────────────

def calculate_turnover(old_weights, new_weights):
    """Total absolute weight change (one-way turnover)."""
    return float(np.sum(np.abs(new_weights - old_weights)))

print("All functions defined.")

## 4. Rolling Backtest Engine

For each test year:
1. Use all available data up to December 28 of the prior year as training
2. Estimate Ledoit-Wolf covariance matrix
3. Optimise weights for each strategy
4. Apply weights to test year, rebalancing within year if specified
5. Deduct transaction costs at each rebalance
6. Record daily returns, final weights, and performance metrics

**Transaction cost assumption:** 5 basis points per unit of one-way turnover (typical for liquid ETFs).  
**Rebalancing:** annual by default — weights are recalculated at the start of each year.

In [ ]:
def rolling_backtest(
    returns,
    tickers,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
    methods=None,
    max_weight=MAX_WEIGHT,
    rebalance="annual",
    transaction_cost_bps=TC_BPS,
    rf=RISK_FREE,
    verbose=True
):
    """
    Rolling-window portfolio backtest across multiple out-of-sample years.

    Parameters
    ----------
    returns : pd.DataFrame       Daily asset returns
    tickers : list               Asset tickers (must match returns columns)
    train_years : int            Number of prior years used for training
    test_years : list            Out-of-sample years to evaluate
    methods : list               Strategies: min_variance, risk_parity, max_sharpe, equal_weight
    max_weight : float           Per-asset weight cap (applies to min_variance only)
    rebalance : str              annual | quarterly | monthly | none
    transaction_cost_bps : float Basis points per unit of one-way turnover
    rf : float                   Risk-free rate for Sharpe calculation
    verbose : bool               Print progress

    Returns
    -------
    port_return_series : dict    {method: pd.Series of daily portfolio returns}
    weights_df : dict            {method: pd.DataFrame of year-end weights}
    metrics_df : dict            {method: pd.DataFrame of annual performance}
    turnover_df : pd.DataFrame   Annual turnover by method
    """
    if methods is None:
        methods = ["min_variance", "risk_parity", "max_sharpe", "equal_weight"]

    all_daily   = {m: [] for m in methods}
    wt_records  = {m: [] for m in methods}
    met_records = {m: [] for m in methods}
    to_records  = []

    for test_year in test_years:
        train_start = f"{test_year - train_years}-01-01"
        train_end   = f"{test_year - 1}-12-28"
        test_start  = f"{test_year}-01-01"
        test_end    = f"{test_year}-12-31"

        ret_train = returns.loc[train_start:train_end, tickers]
        ret_test  = returns.loc[test_start:test_end, tickers]

        if len(ret_train) < 100 or len(ret_test) < 20:
            if verbose: print(f"  {test_year}: insufficient data, skipping")
            continue

        if verbose: print(f"  {test_year}: train={len(ret_train)} obs, test={len(ret_test)} obs")

        # Estimate covariance on full training window
        cov_ann, shrinkage = ledoit_wolf_cov(ret_train)
        exp_ret_ann = ret_train.mean().values * TRADING_DAYS

        # Initial weights
        def get_weights(m, cov, exp_r):
            if m == "min_variance":  return minimize_variance(cov, max_weight)
            elif m == "risk_parity": return optimize_risk_parity(cov)
            elif m == "max_sharpe":  return maximize_sharpe(exp_r, cov, rf)
            elif m == "equal_weight":return np.repeat(1/len(tickers), len(tickers))
            else: raise ValueError(f"Unknown method: {m}")

        init_weights = {m: get_weights(m, cov_ann, exp_ret_ann) for m in methods}

        # Rebalance schedule within test year
        if rebalance == "quarterly":
            rb_set = set(ret_test.resample("Q").first().index.normalize())
        elif rebalance == "monthly":
            rb_set = set(ret_test.resample("M").first().index.normalize())
        elif rebalance == "annual":
            rb_set = {ret_test.index[0].normalize()}
        else:  # none
            rb_set = {ret_test.index[0].normalize()}

        to_row = {"year": test_year}

        for m in methods:
            current_w = init_weights[m].copy()
            day_rets  = []
            total_to  = 0.0

            for i, date in enumerate(ret_test.index):
                tc_cost = 0.0
                if date.normalize() in rb_set:
                    if i > 0:
                        # Re-estimate on expanding window up to prior day
                        expanding_end = ret_test.index[i-1].strftime("%Y-%m-%d")
                        ret_up = returns.loc[train_start:expanding_end, tickers]
                        if len(ret_up) > 100:
                            cov_up, _ = ledoit_wolf_cov(ret_up)
                            exp_up    = ret_up.mean().values * TRADING_DAYS
                            new_w     = get_weights(m, cov_up, exp_up)
                            to        = calculate_turnover(current_w, new_w)
                            tc_cost   = to * transaction_cost_bps / 10000
                            total_to += to
                            current_w = new_w

                dr = float(ret_test.loc[date].values @ current_w) - tc_cost
                day_rets.append(dr)

            series = pd.Series(day_rets, index=ret_test.index, name=m)
            all_daily[m].append(series)
            wt_records[m].append({"year": test_year, **dict(zip(tickers, current_w))})
            mets = performance_metrics(series, rf=rf)
            mets["year"] = test_year
            mets["Ledoit-Wolf alpha"] = shrinkage
            met_records[m].append(mets)
            to_row[m] = total_to

        to_records.append(to_row)

    # Assemble outputs
    port_return_series = {m: pd.concat(all_daily[m]) for m in methods if all_daily[m]}
    weights_df         = {m: pd.DataFrame(wt_records[m]).set_index("year") for m in methods if wt_records[m]}
    metrics_df         = {m: pd.DataFrame(met_records[m]).set_index("year") for m in methods if met_records[m]}
    turnover_df        = pd.DataFrame(to_records).set_index("year") if to_records else pd.DataFrame()

    return port_return_series, weights_df, metrics_df, turnover_df

print("Rolling backtest engine ready.")

## 5. Run the Rolling Backtest

Running four strategies across five out-of-sample years (2019–2023) with annual rebalancing and 5 bps transaction costs.

In [ ]:
METHODS = ["min_variance", "risk_parity", "max_sharpe", "equal_weight"]

print("Running rolling backtest...")
print(f"  Methods:     {METHODS}")
print(f"  Test years:  {TEST_YEARS}")
print(f"  Train years: {TRAIN_YEARS}")
print(f"  Rebalance:   annual")
print(f"  TC:          {TC_BPS} bps\n")

port_rets, wt_hist, met_hist, turnover_df = rolling_backtest(
    returns=returns,
    tickers=TICKERS,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
    methods=METHODS,
    max_weight=MAX_WEIGHT,
    rebalance="annual",
    transaction_cost_bps=TC_BPS,
    verbose=True,
)

print("\nBacktest complete.")

## 6. Annual Performance Tables

In [ ]:
# Annual Sharpe table
sharpe_table = pd.DataFrame({
    LABELS[m]: met_hist[m]["Sharpe"]
    for m in METHODS if m in met_hist
})
print("Annual Sharpe Ratios by Strategy:")
display(sharpe_table.round(4))

In [ ]:
# Annual return table
ret_table = pd.DataFrame({
    LABELS[m]: met_hist[m]["Ann. Return"].map("{:.2%}".format)
    for m in METHODS if m in met_hist
})
print("Annual Returns by Strategy:")
display(ret_table)

In [ ]:
# Annual max drawdown table
dd_table = pd.DataFrame({
    LABELS[m]: met_hist[m]["Max Drawdown"].map("{:.2%}".format)
    for m in METHODS if m in met_hist
})
print("Annual Max Drawdown by Strategy:")
display(dd_table)

In [ ]:
# Full-period aggregate metrics
print("Full-Period Performance (2019-2023 combined):")
agg_rows = {}
for m in METHODS:
    if m in port_rets:
        mets = performance_metrics(port_rets[m])
        agg_rows[LABELS[m]] = mets

agg_df = pd.DataFrame(agg_rows).T
fmt_cols = ["Ann. Return","Ann. Volatility","Max Drawdown","Cum. Return","VaR 95%","CVaR 95%","Hit Rate"]
for col in fmt_cols:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map("{:.2%}".format)
for col in ["Sharpe","Sortino","Calmar"]:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map("{:.4f}".format)

display(agg_df)

## 7. Portfolio Weights Through Time

In [ ]:
print("Portfolio weights by year (end-of-year positions):\n")
for m in METHODS:
    if m in wt_hist:
        print(f"{LABELS[m]}:")
        disp = wt_hist[m].copy()
        for t in TICKERS:
            disp[t] = disp[t].map("{:.2%}".format)
        display(disp)
        print()

In [ ]:
# Weights through time chart
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)
axes = axes.flatten()

for idx, m in enumerate(METHODS):
    if m not in wt_hist:
        continue
    ax = axes[idx]
    w = wt_hist[m]
    x = np.arange(len(w.index))
    bottoms = np.zeros(len(x))
    bar_colors = {"ACWI":"#111111","AGG":"#555555","GLD":"#999999","BSV":"#CCCCCC"}

    for t in TICKERS:
        ax.bar(x, w[t].values, bottom=bottoms, label=t,
               color=bar_colors[t], edgecolor="white", linewidth=0.5)
        bottoms += w[t].values

    ax.set_title(LABELS[m], fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(w.index.astype(str), fontsize=8)
    ax.set_ylabel("Weight")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8, loc="upper right")

fig.suptitle("Portfolio Weights Through Time (Annual Rebalance)", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8. Risk Contribution Analysis

Portfolio weight does not equal risk contribution. BSV holds 40% capital but contributes much less than 40% of total risk because its volatility is low. This section shows the actual risk decomposition of each strategy.

In [ ]:
# Compute risk contributions using the final covariance matrix (2019-2022 window)
ret_final_train = returns.loc["2019-01-01":"2022-12-28", TICKERS]
cov_final, _ = ledoit_wolf_cov(ret_final_train)

print("Risk Contribution Analysis (using 2019-2022 covariance):\n")
rc_data = {}

for m in METHODS:
    if m not in wt_hist:
        continue
    # Use final year weights
    final_w = wt_hist[m].iloc[-1].values
    rc = risk_contributions(final_w, cov_final)
    rc_data[m] = rc
    print(f"{LABELS[m]}:")
    for t, w, r in zip(TICKERS, final_w, rc):
        print(f"  {t:<6}  Weight: {w:6.2%}   Risk Contribution: {r:6.2%}")
    print()

In [ ]:
# Weight vs risk contribution side-by-side chart
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
bar_colors = {"ACWI":"#111111","AGG":"#555555","GLD":"#999999","BSV":"#CCCCCC"}

for idx, m in enumerate(METHODS):
    if m not in wt_hist or m not in rc_data:
        continue
    ax = axes[idx]
    final_w = wt_hist[m].iloc[-1].values
    rc = rc_data[m]
    x = np.arange(len(TICKERS))

    bars1 = ax.bar(x - 0.2, final_w * 100, width=0.35, label="Weight (%)",
                   color=[bar_colors[t] for t in TICKERS], alpha=0.9)
    bars2 = ax.bar(x + 0.2, rc * 100, width=0.35, label="Risk Contribution (%)",
                   color=[bar_colors[t] for t in TICKERS], alpha=0.4,
                   edgecolor=[bar_colors[t] for t in TICKERS], linewidth=1.5)

    ax.set_title(LABELS[m], fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(TICKERS)
    ax.set_ylabel("%")
    ax.set_ylim(0, 55)
    ax.axhline(25, color="grey", linewidth=0.7, linestyle=":")
    ax.grid(axis="y", alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle("Weight vs Risk Contribution by Strategy (Final Year Weights)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Performance Visualisations

In [ ]:
# Chart 1: Rolling cumulative returns
fig, ax = plt.subplots(figsize=(14, 6))
ls_map = {"min_variance":"-","risk_parity":"--","max_sharpe":"-.","equal_weight":":"}

for m in METHODS:
    if m not in port_rets:
        continue
    cum = (1 + port_rets[m]).cumprod()
    ax.plot(cum.index, cum, label=LABELS[m],
            color=COLORS[m], linewidth=2.0, linestyle=ls_map[m])

# Shade each test year
for i, yr in enumerate(TEST_YEARS):
    mask = (port_rets[METHODS[0]].index.year == yr)
    yr_idx = port_rets[METHODS[0]][mask].index
    if len(yr_idx):
        ax.axvspan(yr_idx[0], yr_idx[-1], alpha=0.04, color="grey")
        ax.text(yr_idx[len(yr_idx)//2], 0.75, str(yr), ha="center", va="center",
                fontsize=7.5, color="grey", alpha=0.8)

ax.axhline(1.0, color="black", linewidth=0.6, linestyle=":")
ax.set_title("Rolling Out-of-Sample Cumulative Returns (2019–2023)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value (Start = 1.0)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Rolling drawdowns
fig, ax = plt.subplots(figsize=(14, 5))

for m in METHODS:
    if m not in port_rets:
        continue
    cum = (1 + port_rets[m]).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd * 100, 0, alpha=0.15, color=COLORS[m])
    ax.plot(dd.index, dd * 100, label=LABELS[m],
            color=COLORS[m], linewidth=1.5, linestyle=ls_map[m])

ax.set_title("Rolling Out-of-Sample Drawdowns (2019–2023)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Annual Sharpe ratio by strategy
fig, ax = plt.subplots(figsize=(12, 5))
n_methods = len(METHODS)
x = np.arange(len(TEST_YEARS))
w = 0.18

for i, m in enumerate(METHODS):
    if m not in met_hist:
        continue
    sharpes = met_hist[m]["Sharpe"].reindex(TEST_YEARS).fillna(0)
    offset = (i - n_methods/2 + 0.5) * w
    bars = ax.bar(x + offset, sharpes.values, w, label=LABELS[m],
                  color=COLORS[m], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1.0, color="grey", linewidth=0.6, linestyle="--", alpha=0.5)
ax.set_title("Annual Sharpe Ratio by Strategy (2019–2023)", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in TEST_YEARS])
ax.set_ylabel("Sharpe Ratio")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Annual returns by strategy
fig, ax = plt.subplots(figsize=(12, 5))

for i, m in enumerate(METHODS):
    if m not in met_hist:
        continue
    ann_rets = met_hist[m]["Ann. Return"].reindex(TEST_YEARS).fillna(0)
    offset = (i - n_methods/2 + 0.5) * w
    ax.bar(x + offset, ann_rets.values * 100, w, label=LABELS[m],
           color=COLORS[m], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Annual Return by Strategy (2019–2023)", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in TEST_YEARS])
ax.set_ylabel("Annual Return (%)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Turnover by strategy and year
if not turnover_df.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    t_methods = [m for m in METHODS if m in turnover_df.columns]
    x_to = np.arange(len(turnover_df.index))

    for i, m in enumerate(t_methods):
        offset = (i - len(t_methods)/2 + 0.5) * w
        ax.bar(x_to + offset, turnover_df[m].values * 100, w, label=LABELS[m],
               color=COLORS[m], alpha=0.85, edgecolor="white")

    ax.set_title("Portfolio Turnover by Year (One-Way, %)", fontsize=12, fontweight="bold")
    ax.set_xticks(x_to)
    ax.set_xticklabels([str(y) for y in turnover_df.index])
    ax.set_ylabel("Turnover (%)")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 10. Transaction Cost Sensitivity

How much do transaction costs reduce full-period Sharpe ratio for each strategy?

In [ ]:
tc_scenarios = [0, 5, 10, 25]
tc_results = {}

print("Running transaction cost sensitivity analysis...")
for tc in tc_scenarios:
    _, _, met_tc, _ = rolling_backtest(
        returns=returns, tickers=TICKERS, train_years=TRAIN_YEARS,
        test_years=TEST_YEARS, methods=METHODS,
        max_weight=MAX_WEIGHT, rebalance="annual",
        transaction_cost_bps=tc, verbose=False
    )
    tc_results[tc] = {m: met_tc[m]["Sharpe"].mean() for m in METHODS if m in met_tc}
    print(f"  TC = {tc:2d} bps  done")

tc_df = pd.DataFrame(tc_results).T
tc_df.index.name = "TC (bps)"
tc_df.columns = [LABELS[m] for m in tc_df.columns]
print("\nAverage Annual Sharpe Ratio by Transaction Cost Assumption:")
display(tc_df.round(4))

In [ ]:
# TC sensitivity chart
fig, ax = plt.subplots(figsize=(10, 5))
for m in METHODS:
    col = LABELS[m]
    if col in tc_df.columns:
        ax.plot(tc_df.index, tc_df[col], label=col,
                color=COLORS[m], linewidth=2, marker="o", markersize=5,
                linestyle=ls_map[m])

ax.set_title("Average Sharpe Ratio vs Transaction Cost Assumption", fontsize=12, fontweight="bold")
ax.set_xlabel("Transaction Cost (bps)")
ax.set_ylabel("Average Annual Sharpe Ratio (2019-2023)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Rebalancing Frequency Sensitivity

How does rebalancing frequency affect performance for the Minimum Variance strategy?

In [ ]:
rebalance_scenarios = ["none", "annual", "quarterly", "monthly"]
rb_results = {}

print("Running rebalancing frequency analysis (Min Variance only)...")
for rb in rebalance_scenarios:
    _, _, met_rb, to_rb = rolling_backtest(
        returns=returns, tickers=TICKERS, train_years=TRAIN_YEARS,
        test_years=TEST_YEARS, methods=["min_variance"],
        max_weight=MAX_WEIGHT, rebalance=rb,
        transaction_cost_bps=TC_BPS, verbose=False
    )
    if "min_variance" in met_rb:
        mets = met_rb["min_variance"]
        rb_results[rb] = {
            "Avg Sharpe":       mets["Sharpe"].mean(),
            "Avg Return":       mets["Ann. Return"].mean(),
            "Avg Volatility":   mets["Ann. Volatility"].mean(),
            "Avg Max Drawdown": mets["Max Drawdown"].mean(),
            "Avg Turnover":     to_rb["min_variance"].mean() if "min_variance" in to_rb.columns else 0.0,
        }
    print(f"  {rb:<12} done")

rb_df = pd.DataFrame(rb_results).T
rb_df.index.name = "Rebalance"
for col in ["Avg Return","Avg Volatility","Avg Max Drawdown","Avg Turnover"]:
    rb_df[col] = rb_df[col].map("{:.2%}".format)
rb_df["Avg Sharpe"] = rb_df["Avg Sharpe"].map("{:.4f}".format)
print("\nMin Variance: Impact of Rebalancing Frequency")
display(rb_df)

## 12. Summary

### What the rolling backtest shows

**Strategy performance varies substantially by year.** 2022 was a difficult year for all four strategies because equities and bonds fell simultaneously due to the Federal Reserve rate shock. 2023 rewarded strategies with higher ACWI exposure.

**Minimum Variance is consistent but conservative.** It consistently has the lowest volatility and shallowest drawdowns. In bull years it lags on return. In bear years (2022) it provides downside protection relative to Equal Weight.

**Risk Parity is the most balanced strategy.** By equalising risk contribution rather than capital weight, it avoids over-concentration in any single asset's volatility regime. It tends to produce better Sharpe ratios than Minimum Variance in up years.

**Maximum Sharpe is the most sensitive strategy.** Small changes in expected return estimates cause large weight swings year to year. This is the core finding from Version 1 confirmed across the rolling window.

**Equal Weight underperforms on volatility and drawdown** but can outperform on raw return in years when all assets rally (2019, 2021, 2023).

### Transaction costs

At 5 bps per unit of turnover, annual rebalancing costs are negligible for all strategies. Only monthly rebalancing at 25 bps produces meaningful performance drag.

### Key conclusions

1. No single strategy dominates across all five years.
2. Minimum Variance offers the most consistent downside protection.
3. Risk Parity is the best balance between diversification and return.
4. Maximum Sharpe is not recommended without a principled return forecasting framework.
5. A 40% cap on Minimum Variance is essential — unconstrained it collapses to a single asset.

In [ ]:
# Final summary table
print("=" * 70)
print("FULL-PERIOD SUMMARY (2019-2023, annual rebalancing, 5 bps TC)")
print("=" * 70)

summary_rows = []
for m in METHODS:
    if m in port_rets:
        mets = performance_metrics(port_rets[m])
        summary_rows.append({
            "Strategy":         LABELS[m],
            "Cum. Return":      f"{mets['Cum. Return']:.2%}",
            "Ann. Return":      f"{mets['Ann. Return']:.2%}",
            "Volatility":       f"{mets['Ann. Volatility']:.2%}",
            "Sharpe":           f"{mets['Sharpe']:.4f}",
            "Sortino":          f"{mets['Sortino']:.4f}",
            "Max Drawdown":     f"{mets['Max Drawdown']:.2%}",
            "Calmar":           f"{mets['Calmar']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows).set_index("Strategy")
display(summary_df)

## Disclaimer

This notebook is for academic and educational purposes only. It does not constitute investment advice or a recommendation to buy or sell any security.